1) I have pdf files in a folder. The program reads PDF files and stores it in vectorDB

2) The user asks a question, the vector DB return relevant chunks related to the query

3) I pass the retrieved results + query + prompt to the LLM

Here I am using OpenAI embedding model to create embeddings

In [1]:
# Check python version: Should be 3.10
!python --version

Python 3.10.11


In [2]:
# This took 5 minutes

!pip install langchain==0.2.14 langchain-community==0.2.12 langchain-openai==0.1.25 langchain-text-splitters 

  Using cached langchain-0.2.14-py3-none-any.whl (997 kB)
  Using cached langchain_community-0.2.12-py3-none-any.whl (2.3 MB)
  Using cached langchain_openai-0.1.25-py3-none-any.whl (51 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl (35 kB)
  Using cached aiohttp-3.13.3-cp310-cp310-win_amd64.whl (456 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl (25 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl (311 kB)
  Using cached async_timeout-4.0.3-py3-none-any.whl (5.7 kB)
  Using cached sqlalchemy-2.0.48-cp310-cp310-win_amd64.whl (2.1 MB)
  Using cached tenacity-8.5.0-py3-none-any.whl (28 kB)
  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl (15.8 MB)
  Using cached langchain_core-0.2.43-py3-none-any.whl (397 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
  Using cached tiktoken-0.12.0-cp310-cp310-win_amd64.whl (879 kB)
  Using cached openai-1.109.1-py3-none-any.whl 


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install pypdf

  Using cached pypdf-6.9.1-py3-none-any.whl (333 kB)



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Now restart the kernel

In [4]:
import langchain
print(langchain.__version__)

0.2.14


In [5]:
!pip install pinecone
!pip install python-dotenv

  Using cached pinecone-8.1.0-py3-none-any.whl (742 kB)
  Using cached pinecone_plugin_assistant-3.0.2-py3-none-any.whl (280 kB)
  Using cached pinecone_plugin_interface-0.0.7-py3-none-any.whl (6.2 kB)



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# import Libraries

import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from pinecone import Pinecone, ServerlessSpec

In [7]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads variables from .env

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")

# Phase 1: Populate the Vector DB

In [9]:
## Read the document
def read_doc(directory):
    file_loader=PyPDFDirectoryLoader(directory)
    documents=file_loader.load()
    return documents

docs=read_doc("documents-0-test")
print("Number of pages:",len(docs))
print("Type of object:", type(docs[0]))

Number of pages: 2
Type os object: <class 'langchain_core.documents.base.Document'>


In [10]:
# Show me pages
import os

for doc in docs:
    file_path = doc.metadata.get("source", "")
    file_name = os.path.basename(file_path)
    page = doc.metadata.get("page")
    
    print("File:",file_name)
    print("Page:", page)
    print("First few snippet:", doc.page_content[:200])
    print("---------")

File: Quantum Mechanics-LoxfordAcademy-small.pdf
Page: 0
First few snippet: Quantum Mechanics: A Grand Tour of the Tiny Universe 
Executive Summary: Quantum mechanics is the branch of physics governing the very 
small – atoms, electrons, photons, and other particles. As one s
---------
File: Quantum Mechanics-LoxfordAcademy-small.pdf
Page: 1
First few snippet: Loxford Academy: 
Loxford Academy was founded by Alberto Graciaso Negurasa and Rohit Salvadore Lomax. 
Its address is 5432, Deru Valley on planet Mars.  Loxford Academy has 34,000 employees 
and has a
---------


In [11]:
# Split pdf into pages and split pages into chunks

def chunk_data(docs, chunk_size=800, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_docs = []

    for doc in docs:
        chunks = splitter.split_text(doc.page_content)

        for chunk in chunks:
            chunked_docs.append({
                "text": chunk,
                "metadata": {
                    "source": doc.metadata.get("source", ""),
                    "page": doc.metadata.get("page")
                }
            })

    return chunked_docs

documents=chunk_data(docs=docs)
print("Total chunks =", len(documents))
# print(documents[0])

for d in documents:
    print("text:", d['text'])
    print("metadata:", d['metadata'])
    print("------------")

Total chunks = 4
text: Quantum Mechanics: A Grand Tour of the Tiny Universe 
Executive Summary: Quantum mechanics is the branch of physics governing the very 
small – atoms, electrons, photons, and other particles. As one source notes, “Quantum 
mechanics is the fundamental physical theory that describes the behavior of matter and of 
light; its unusual characteristics typically occur…at and below the scale of atoms.”[1]. In 
other words, it’s how the universe works when you zoom in far beyond everyday 
experience. It arose in the early 1900s to explain puzzling experiments (like blackbody 
radiation and the photoelectric effect) that classical physics couldn’t[2]. This report gives a 
concise introduction: we’ll define quantum mechanics, outline its history with a timeline,
metadata: {'source': 'documents-0-test\\Quantum Mechanics-LoxfordAcademy-small.pdf', 'page': 0}
------------
text: concise introduction: we’ll define quantum mechanics, outline its history with a timeline, 
meet tw

## Insert the above embedding chunks to Pinecone vectorDB

### Create index

In [12]:
## Vector Search DB In Pinecone
pc = Pinecone(
        api_key=PINECONE_API_KEY
    )

index_name = 'langchainvector' # needs to be created first

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric='euclidean',
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )


index = pc.Index(index_name)
print("==============",index.describe_index_stats())

============== {'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '154',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 22 Mar 2026 06:44:19 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '38',
                                    'x-pinecone-request-latency-ms': '37',
                                    'x-pinecone-response-duration-ms': '39'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'euclidean',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


### Use OpenAI embedding model
- I could have used some other embedding models too.

In [13]:
## Embedding Technique Of OPENAI
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small", # cheapest model
    api_key=OPENAI_API_KEY
)

print(embeddings.model)
print("embeddings=",embeddings) # info about embedding

text-embedding-3-small
embeddings= client=<openai.resources.embeddings.Embeddings object at 0x00000149D01347C0> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x00000149D2396B90> model='text-embedding-3-small' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True


In [14]:

def get_embedding(text):
    embedding = embeddings.embed_query(doc.page_content)
    return embedding


def populate_vectorDB(documents):

    for i, doc in enumerate(documents):

        # Extract text and metadata
        text = doc.get("text", "")
        metadata = doc.get("metadata", {})

        file_path = metadata.get("source", "")
        file_name = os.path.basename(file_path)
        page_number = metadata.get("page", 0)

        # Generate embedding
        embedding = embeddings.embed_query(text)

        # Insert into Pinecone
        index.upsert([{
            "id": str(i),
            "values": embedding,
            "metadata": {
                "text": text,
                "source": file_name,
                "page": page_number,
                "chunk_id": i
            }
        }])


# call the function
populate_vectorDB(documents)
print("Documents inserted into vector DB")

Documents inserted into vector DB


# Phase 2: The user query

In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

# Retrieve top 2 results similar to query from VectorDB

In [16]:
def retrieve_query_results_from_vectorDB(query, k=2):
    matching_results = []

    query_embedding = get_embedding(query)

    query_results = index.query(
        vector=query_embedding,
        top_k=k,
        include_metadata=True
    )

    for match in query_results['matches']:
        page_content = match['metadata']['text']
        metadata = {
        "source": match['metadata'].get('source'),
        "page": match['metadata'].get('page'),
        "chunk_id": match['metadata'].get('chunk_id')
        }

        matching_results.append({
            "page_content": page_content,
            "metadata": metadata
        })

    return matching_results


In [17]:
query = "Who are the founders of Loxford Academy?"
matching_results = retrieve_query_results_from_vectorDB(query)

for m in matching_results:
    print(m)
    print("------")

{'page_content': 'Loxford Academy: \nLoxford Academy was founded by Alberto Graciaso Negurasa and Rohit Salvadore Lomax. \nIts address is 5432, Deru Valley on planet Mars.  Loxford Academy has 34,000 employees \nand has a revenue of 99 million dollars. Its employee workforce consists of citizen from  \nAustralia, India, USA, Europe, and many other countries. Its headquarter is in Honda Valley \nlocated on Mars.', 'metadata': {'source': 'Quantum Mechanics-LoxfordAcademy-small.pdf', 'page': 1, 'chunk_id': 3}}
------
{'page_content': 'Quantum Mechanics: A Grand Tour of the Tiny Universe \nExecutive Summary: Quantum mechanics is the branch of physics governing the very \nsmall – atoms, electrons, photons, and other particles. As one source notes, “Quantum \nmechanics is the fundamental physical theory that describes the behavior of matter and of \nlight; its unusual characteristics typically occur…at and below the scale of atoms.”[1]. In \nother words, it’s how the universe works when you 

## Create a prompt template

In [19]:
# Create Chain

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Give instructions in prompt so it does NOT hallucinate
prompt = ChatPromptTemplate.from_template(
    """
    Answer ONLY from the context.
    If the answer is not in the context, say "I don't know".

    Context:
    {context}

    Question:
    {user_query}
    """
)


chain = create_stuff_documents_chain(llm, prompt)

In [22]:
from langchain.schema import Document

def retrieve_answers_from_LLM(query):
    matching_results = retrieve_query_results_from_vectorDB(query)

    # Convert
    matching_docs = [
        Document(page_content=doc['page_content'])
        for doc in matching_results
    ]
    
    response = chain.invoke({
        "user_query": query,
        "context": matching_docs
    })

    # 🔹 Keep metadata separately (for you)
    sources = []
    for doc in matching_results:
        sources.append({
            "source": doc['metadata'].get("source"),
            "page":   doc['metadata'].get("page"),
            "chunk":  doc['metadata'].get("chunk_id")
        })
        
    return response, sources


In [23]:
# Example query
query = "Who are the founders of Loxford Academy ?"

answer, sources = retrieve_answers_from_LLM(query)

print("Answer:\n", answer)

print("\nSources:")
for s in sources:
    print(s)

Answer:
 The founders of Loxford Academy are Alberto Graciaso Negurasa and Rohit Salvadore Lomax.

Sources:
{'source': 'Quantum Mechanics-LoxfordAcademy-small.pdf', 'page': 1, 'chunk': 3}
{'source': 'Quantum Mechanics-LoxfordAcademy-small.pdf', 'page': 0, 'chunk': 0}


# Delete Pinecone Index

In [24]:
pc.delete_index(index_name)

In [25]:
for idx in pc.list_indexes():
    print(idx)